# Extended Model Taxonomy Analysis
## Rough-Regime Volatility Engine — Multi-Model Framework

This notebook implements and visualises four canonical stochastic volatility models alongside the existing Volterra/HMM infrastructure:

1. **Heston (1993)** — Mean-reverting CIR variance with leverage effect
2. **SABR (Hagan et al., 2002)** — Stochastic-alpha-beta-rho; the market standard for FI/FX smiles
3. **Local Volatility (Dupire, 1994)** — Deterministic $\sigma(t, S)$ calibrated to a full market surface
4. **Hawkes Jump-Diffusion** — Self-exciting point process for panic and contagion dynamics

**Paper reference:** *Rough Volatility Arbitrage under Markov Regime Switching: A Volterra Process Approach with HMM Risk Gating*

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib import gridspec
from scipy.stats import norm
from scipy.optimize import brentq, minimize
from scipy.interpolate import RectBivariateSpline
import warnings
warnings.filterwarnings('ignore')

# Global plot style
plt.rcParams.update({
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'serif',
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'legend.fontsize': 9,
})

RNG = np.random.default_rng(42)
print('Environment ready.')

---
## Section 1 — Heston Stochastic Volatility Model

The Heston (1993) model specifies joint dynamics for the spot $S_t$ and its instantaneous variance $V_t$:

$$dS_t = r S_t \, dt + \sqrt{V_t} \, S_t \, dW_t^S$$

$$dV_t = \kappa(\theta - V_t)\,dt + \xi\sqrt{V_t}\,dW_t^V, \qquad d\langle W^S, W^V\rangle_t = \rho\,dt$$

The **Feller condition** $2\kappa\theta > \xi^2$ ensures $V_t > 0$ almost surely. Negative $\rho$ induces the empirically observed **leverage effect**: spot down, vol up.

In [ ]:
# ── Heston Parameters ─────────────────────────────────────────────────────
HESTON_PARAMS = dict(
    S0    = 100.0,   # initial spot
    V0    = 0.04,    # initial variance  (σ₀ ≈ 20%)
    kappa = 2.0,     # mean-reversion speed
    theta = 0.04,    # long-run variance   (σ_∞ ≈ 20%)
    xi    = 0.3,     # vol-of-vol
    rho   = -0.7,    # spot-vol correlation (leverage)
    r     = 0.05,    # risk-free rate
)

print(f"Feller condition: 2κθ={2*HESTON_PARAMS['kappa']*HESTON_PARAMS['theta']:.4f} > ξ²={HESTON_PARAMS['xi']**2:.4f} → "
      f"{'SATISFIED' if 2*HESTON_PARAMS['kappa']*HESTON_PARAMS['theta'] > HESTON_PARAMS['xi']**2 else 'VIOLATED'}")

In [ ]:
def heston_simulate_paths(S0, V0, kappa, theta, xi, rho, r,
                          T=1.0, n_steps=252, n_paths=5000, seed=42):
    """
    Simulate Heston paths via Euler-Milstein discretisation with
    full truncation (V absorbing at 0) for the variance process.
    """
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    sqrt_dt = np.sqrt(dt)

    S = np.full(n_paths, S0)
    V = np.full(n_paths, V0)

    S_paths = np.zeros((n_paths, n_steps + 1))
    V_paths = np.zeros((n_paths, n_steps + 1))
    S_paths[:, 0] = S
    V_paths[:, 0] = V

    sqrt1mrho2 = np.sqrt(1.0 - rho**2)

    for i in range(n_steps):
        Z1 = rng.standard_normal(n_paths)
        Z2 = rng.standard_normal(n_paths)
        dW_S = sqrt_dt * Z1
        dW_V = sqrt_dt * (rho * Z1 + sqrt1mrho2 * Z2)

        sqrtV = np.sqrt(np.maximum(V, 0.0))

        # Milstein correction term for variance
        V_new = V + kappa * (theta - V) * dt + xi * sqrtV * dW_V \
                + 0.25 * xi**2 * (dW_V**2 - dt)  # Milstein
        V = np.maximum(V_new, 0.0)   # full truncation

        S = S * np.exp((r - 0.5 * V) * dt + sqrtV * dW_S)

        S_paths[:, i + 1] = S
        V_paths[:, i + 1] = V

    return S_paths, V_paths


def bs_iv(price, S, K, T, r, option_type='call'):
    """Implied volatility via Brent root-finding."""
    intrinsic = max((S - K) if option_type == 'call' else (K - S), 0.0)
    if price <= intrinsic + 1e-10:
        return np.nan
    def objective(sigma):
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        if option_type == 'call':
            return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2) - price
        else:
            return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1) - price
    try:
        return brentq(objective, 1e-6, 5.0, xtol=1e-8)
    except Exception:
        return np.nan


def heston_vol_surface(params, strikes_pct, maturities):
    """
    Build a Heston implied-volatility surface via Monte Carlo pricing
    for each (T, K) node.
    """
    surface = np.full((len(maturities), len(strikes_pct)), np.nan)

    for i, T in enumerate(maturities):
        n_steps = max(int(T * 252), 50)
        S_paths, _ = heston_simulate_paths(
            **params, T=T, n_steps=n_steps, n_paths=4000)
        S_T = S_paths[:, -1]
        discount = np.exp(-params['r'] * T)

        for j, kpct in enumerate(strikes_pct):
            K = params['S0'] * kpct
            payoffs = np.maximum(S_T - K, 0.0)
            price = discount * np.mean(payoffs)
            surface[i, j] = bs_iv(price, params['S0'], K, T, params['r'])

    return surface


# Build surface
strikes_pct = np.array([0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20])
maturities   = np.array([0.083, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0])  # years

print('Computing Heston vol surface (this takes ~30s)...')
heston_surface = heston_vol_surface(HESTON_PARAMS, strikes_pct, maturities)
print('Done.')

In [ ]:
# ── Figure 1: Heston Volatility Surface ───────────────────────────────────
fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 2, width_ratios=[1.4, 1])

# 3-D surface
ax3d = fig.add_subplot(gs[0], projection='3d')
K_grid, T_grid = np.meshgrid(strikes_pct, maturities)
ax3d.plot_surface(K_grid, T_grid, heston_surface * 100,
                  cmap='RdYlGn_r', edgecolor='none', alpha=0.88)
ax3d.set_xlabel('Moneyness $K/S_0$', labelpad=8)
ax3d.set_ylabel('Maturity $T$ (yr)', labelpad=8)
ax3d.set_zlabel('Implied Vol (%)')
ax3d.set_title(r'Heston IV Surface ($\rho=-0.7$, leverage effect)')
ax3d.view_init(elev=28, azim=-55)

# Variance paths
ax2 = fig.add_subplot(gs[1])
S_ex, V_ex = heston_simulate_paths(**HESTON_PARAMS, T=1.0, n_steps=252, n_paths=200, seed=99)
t_axis = np.linspace(0, 1, 253)
for k in range(50):
    ax2.plot(t_axis, np.sqrt(V_ex[k]) * 100, lw=0.6,
             color=cm.plasma(0.2 + 0.6 * k / 50), alpha=0.55)
ax2.axhline(np.sqrt(HESTON_PARAMS['theta']) * 100, ls='--', lw=1.5,
            color='navy', label=r'$\sqrt{\theta}$ (long-run vol)')
ax2.set_xlabel('Time (years)')
ax2.set_ylabel('Instantaneous Vol (%)')
ax2.set_title('Heston Variance Paths (50 samples)')
ax2.legend()

plt.tight_layout()
plt.savefig('heston_surface.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → heston_surface.png')

---
## Section 2 — SABR Model

The SABR model (Hagan et al., 2002) is the market-standard framework for interest-rate derivatives and FX options:

$$dF_t = \sigma_t F_t^\beta \, dW_t^1$$
$$d\sigma_t = \alpha \, \sigma_t \, dW_t^2, \qquad d\langle W^1, W^2 \rangle_t = \rho \, dt$$

The Hagan et al. (2002) **closed-form approximation** for implied Black volatility:

$$\sigma_{\text{imp}}(F, K) \approx \frac{\alpha}{(FK)^{(1-\beta)/2}} \cdot \frac{z}{\chi(z)} \cdot \left[1 + \left(\frac{(1-\beta)^2}{24}\frac{\alpha^2}{(FK)^{1-\beta}} + \frac{\rho\beta\nu\alpha}{4(FK)^{(1-\beta)/2}} + \frac{2-3\rho^2}{24}\nu^2\right)T + \cdots\right]$$

where $z = \frac{\nu}{\alpha}(FK)^{(1-\beta)/2}\ln(F/K)$ and $\chi(z)=\ln\frac{\sqrt{1-2\rho z+z^2}+z-\rho}{1-\rho}$.

In [ ]:
def sabr_implied_vol(F, K, T, alpha, beta, rho, nu):
    """
    Hagan et al. (2002) SABR implied (Black) volatility approximation.
    Handles ATM (F=K) via L'Hôpital limit.
    """
    K = np.atleast_1d(np.asarray(K, dtype=float))
    iv = np.empty_like(K)

    for idx, k in enumerate(K):
        if abs(F - k) < 1e-8:           # ATM limit
            FK_mid = F
            A = alpha / (FK_mid ** (1 - beta))
            B = 1 + ((1-beta)**2/24 * alpha**2 / FK_mid**(2*(1-beta))
                     + rho*beta*nu*alpha / (4 * FK_mid**((1-beta)))
                     + (2 - 3*rho**2)/24 * nu**2) * T
            iv[idx] = A * B
        else:
            FK_mid  = np.sqrt(F * k)
            log_FK  = np.log(F / k)
            one_m_b = 1 - beta

            z = (nu / alpha) * FK_mid**one_m_b * log_FK
            chi_z = np.log((np.sqrt(1 - 2*rho*z + z**2) + z - rho) / (1 - rho))

            numer = alpha
            denom1 = FK_mid**one_m_b * (1 + one_m_b**2/24 * log_FK**2
                                         + one_m_b**4/1920 * log_FK**4)

            factor1 = z / chi_z
            factor2 = 1 + (one_m_b**2/24 * alpha**2 / FK_mid**(2*one_m_b)
                           + rho*beta*nu*alpha / (4*FK_mid**one_m_b)
                           + (2 - 3*rho**2)/24 * nu**2) * T

            iv[idx] = (numer / denom1) * factor1 * factor2

    return iv.squeeze()


# ── Parameter sets to demonstrate smile shape sensitivity ─────────────────
F   = 0.03          # ATM forward rate  (3%, representative swaption)
T   = 1.0           # 1-year maturity
strikes_ir = np.linspace(0.005, 0.065, 80)

sabr_scenarios = {
    r'Symmetric  ($\rho=0$)':       dict(alpha=0.025, beta=0.5, rho=0.00, nu=0.4),
    r'Skew down  ($\rho=-0.4$)':    dict(alpha=0.025, beta=0.5, rho=-0.4, nu=0.4),
    r'High VolVol ($\nu=0.8$)':     dict(alpha=0.025, beta=0.5, rho=-0.2, nu=0.8),
    r'$\beta=1$ (log-normal)':       dict(alpha=0.020, beta=1.0, rho=-0.3, nu=0.4),
}

for name, p in sabr_scenarios.items():
    iv = sabr_implied_vol(F, strikes_ir, T, **p)
    print(f'{name:30s}  ATM IV = {sabr_implied_vol(F, F, T, **p):.4f}')

In [ ]:
# ── Figure 2: SABR Implied Volatility Smile ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colours = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd']
ax = axes[0]
for (name, p), c in zip(sabr_scenarios.items(), colours):
    iv = sabr_implied_vol(F, strikes_ir, T, **p) * 100
    ax.plot(strikes_ir * 100, iv, lw=2, color=c, label=name)
ax.axvline(F * 100, ls=':', lw=1.2, color='grey', label='ATM forward')
ax.set_xlabel('Strike (%)'); ax.set_ylabel('Implied Volatility (%)')
ax.set_title('SABR Implied Vol Smile/Skew (T=1yr, F=3%)')
ax.legend(fontsize=8); ax.grid(alpha=0.25)

# Term-structure of ATM vol
ax2 = axes[1]
Ts  = np.array([0.25, 0.5, 1.0, 2.0, 5.0, 10.0])
base_p = dict(alpha=0.025, beta=0.5, rho=-0.3, nu=0.4)
for nu_val, c in zip([0.2, 0.4, 0.8], ['#1f77b4', '#2ca02c', '#d62728']):
    p = dict(**base_p); p['nu'] = nu_val
    atm_ivs = [sabr_implied_vol(F, F, t, **p) * 100 for t in Ts]
    ax2.plot(Ts, atm_ivs, lw=2, marker='o', ms=5, color=c,
             label=fr'$\nu={nu_val}$')
ax2.set_xlabel('Maturity $T$ (years)')
ax2.set_ylabel('ATM Implied Vol (%)')
ax2.set_title('SABR ATM Vol Term Structure')
ax2.legend(); ax2.grid(alpha=0.25)

plt.tight_layout()
plt.savefig('sabr_smile.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → sabr_smile.png')

---
## Section 3 — Local Volatility (Dupire)

Dupire (1994) showed that any arbitrage-free implied volatility surface $\sigma_{\text{imp}}(T, K)$ can be reproduced by a **deterministic local volatility function** $\sigma_{\text{loc}}(t, S)$ satisfying:

$$\sigma_{\text{loc}}^2(T, K) = \frac{\partial_T C + r K \, \partial_K C}{\frac{1}{2} K^2 \partial_{KK} C}$$

where $C(T, K)$ is the market call price. This is **exact**—the local vol model by construction fits every observed option price simultaneously, making it the canonical hedge for path-dependent exotics.

In [ ]:
def call_price_bs(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0:
        return max(S - K, 0.0)
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)


def synthetic_iv_surface(S0, r, T_grid, K_grid):
    """
    Generate a synthetic implied vol surface using a parametric form that
    blends skew (SABR-style) with a realistic ATM term structure.
    Mimics SPX-like vol surface.
    """
    n_T, n_K = len(T_grid), len(K_grid)
    iv = np.zeros((n_T, n_K))

    for i, T in enumerate(T_grid):
        atm_vol = 0.18 + 0.03 * np.exp(-2*T)          # slight term structure
        skew    = -0.12 / np.sqrt(T)                   # skew steepens at short T
        kurt    =  0.04 / T                             # smile convexity
        for j, K in enumerate(K_grid):
            x = np.log(K / S0) / (atm_vol * np.sqrt(T))  # moneyness
            iv[i, j] = atm_vol + skew * x + kurt * x**2
            iv[i, j] = max(iv[i, j], 0.005)             # floor at 50bps
    return iv


def dupire_local_vol(S0, r, T_grid, K_grid, iv_surface):
    """
    Numerical Dupire formula via finite differences on call prices.
    Returns local vol surface on the interior of (T, K) grid.
    """
    # Build call price surface  C(T, K)
    C = np.zeros_like(iv_surface)
    for i, T in enumerate(T_grid):
        for j, K in enumerate(K_grid):
            C[i, j] = call_price_bs(S0, K, T, r, iv_surface[i, j])

    n_T, n_K = len(T_grid), len(K_grid)
    lv = np.full((n_T, n_K), np.nan)

    # Interior points only
    for i in range(1, n_T - 1):
        dT  = T_grid[i+1] - T_grid[i-1]
        for j in range(1, n_K - 1):
            K   = K_grid[j]
            dK  = K_grid[j+1] - K_grid[j-1]
            dK2 = K_grid[j+1] - K_grid[j]  # non-uniform grid handling

            dC_dT   = (C[i+1, j] - C[i-1, j]) / dT
            dC_dK   = (C[i, j+1] - C[i, j-1]) / dK
            d2C_dK2 = (C[i, j+1] - 2*C[i, j] + C[i, j-1]) / (0.5*dK)**2

            numer = dC_dT + r * K * dC_dK
            denom = 0.5 * K**2 * d2C_dK2

            if denom > 1e-10 and numer > 0:
                lv[i, j] = np.sqrt(numer / denom)
            else:
                lv[i, j] = iv_surface[i, j]  # fallback to market IV

    # Fill edges via nearest-neighbour
    lv[0, :]  = lv[1, :]
    lv[-1, :] = lv[-2, :]
    lv[:, 0]  = lv[:, 1]
    lv[:, -1] = lv[:, -2]

    return lv


# Build grids
S0_lv = 100.0
r_lv  = 0.05
T_grid = np.linspace(0.05, 2.0, 30)
K_grid = np.linspace(70, 135, 40)

iv_mkt = synthetic_iv_surface(S0_lv, r_lv, T_grid, K_grid)
lv_mkt = dupire_local_vol(S0_lv, r_lv, T_grid, K_grid, iv_mkt)

print(f'Local vol range: [{np.nanmin(lv_mkt)*100:.1f}%, {np.nanmax(lv_mkt)*100:.1f}%]')
print(f'Market IV range: [{iv_mkt.min()*100:.1f}%, {iv_mkt.max()*100:.1f}%]')

In [ ]:
# ── Figure 3: Local Volatility Surface ────────────────────────────────────
fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 2)

TT, KK = np.meshgrid(T_grid, K_grid, indexing='ij')

ax1 = fig.add_subplot(gs[0], projection='3d')
ax1.plot_surface(KK, TT, iv_mkt * 100, cmap='viridis', edgecolor='none', alpha=0.85)
ax1.set_xlabel('Strike $K$'); ax1.set_ylabel('Maturity $T$ (yr)')
ax1.set_zlabel('Implied Vol (%)')
ax1.set_title('Market Implied Vol Surface (synthetic)')
ax1.view_init(elev=30, azim=-50)

ax2 = fig.add_subplot(gs[1], projection='3d')
surf = ax2.plot_surface(KK, TT, lv_mkt * 100, cmap='plasma', edgecolor='none', alpha=0.85)
ax2.set_xlabel('Strike $K$'); ax2.set_ylabel('Maturity $T$ (yr)')
ax2.set_zlabel('Local Vol $\\sigma(T,K)$ (%)')
ax2.set_title('Dupire Local Volatility Surface (calibrated)')
ax2.view_init(elev=30, azim=-50)

plt.tight_layout()
plt.savefig('dupire_local_vol.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → dupire_local_vol.png')

---
## Section 4 — Hawkes Jump-Diffusion (Self-Exciting Process)

A **Hawkes process** is a self-exciting point process in which each jump increases the conditional intensity, modeling the contagion and clustering observed during market crises:

$$\lambda(t) = \mu + \sum_{t_i < t} \phi(t - t_i), \qquad \phi(s) = \alpha_H \, e^{-\beta_H s}$$

- $\mu$ — baseline intensity (background jump rate)
- $\alpha_H$ — excitation (each jump raises intensity by $\alpha_H$)
- $\beta_H$ — decay rate (intensity mean-reverts at rate $\beta_H$)
- Stationarity condition: $\alpha_H / \beta_H < 1$

The combined **Hawkes Jump-Diffusion** price process is:

$$dS_t = \mu_S S_t \, dt + \sigma_S S_t \, dW_t + S_{t^-}(e^J - 1) \, dN_t$$

where $J \sim \mathcal{N}(\mu_J, \sigma_J^2)$ and $N_t$ is the Hawkes counting process.

In [ ]:
def simulate_hawkes(mu, alpha_h, beta_h, T, seed=42):
    """
    Simulate a univariate Hawkes process via Ogata's thinning algorithm.
    Returns array of event times in [0, T].
    """
    rng = np.random.default_rng(seed)
    events = []
    t = 0.0

    # Current intensity accumulator
    lam_star = mu

    while t < T:
        # Draw inter-arrival from upper-bound intensity
        u = rng.uniform()
        w = -np.log(u) / lam_star
        t_candidate = t + w
        if t_candidate > T:
            break

        # Compute true intensity at candidate time
        lam_true = mu + sum(alpha_h * np.exp(-beta_h * (t_candidate - s))
                            for s in events)

        # Thinning acceptance
        if rng.uniform() <= lam_true / lam_star:
            events.append(t_candidate)
            lam_star = lam_true + alpha_h   # intensity jumps at event
        else:
            lam_star = lam_true

        t = t_candidate

    return np.array(events)


def hawkes_intensity(events, mu, alpha_h, beta_h, t_grid):
    """Evaluate λ(t) on a time grid given event history."""
    lam = np.full_like(t_grid, mu)
    for ev in events:
        mask = t_grid > ev
        lam[mask] += alpha_h * np.exp(-beta_h * (t_grid[mask] - ev))
    return lam


def simulate_hawkes_jump_diffusion(S0, mu_s, sigma_s, mu_J, sigma_J,
                                   hawkes_mu, alpha_h, beta_h,
                                   T=1.0, dt=1/252, seed=42):
    """
    Simulate a Hawkes jump-diffusion price path.
    """
    rng = np.random.default_rng(seed)
    n   = int(T / dt)
    t   = np.linspace(0, T, n + 1)
    S   = np.zeros(n + 1); S[0] = S0

    # Pre-simulate Hawkes events
    events = simulate_hawkes(hawkes_mu, alpha_h, beta_h, T, seed=seed)

    # Assign events to time steps
    event_steps = set(np.searchsorted(t, events, side='right') - 1)

    log_S = np.log(S0)
    log_S_path = [log_S]

    # Compute compensator for martingale correction
    kappa_comp = np.exp(mu_J + 0.5 * sigma_J**2) - 1
    lambda_bar  = hawkes_mu / (1 - alpha_h / beta_h)   # stationary mean intensity

    for i in range(n):
        dW  = rng.standard_normal() * np.sqrt(dt)
        drift = (mu_s - 0.5*sigma_s**2 - lambda_bar*kappa_comp) * dt
        diffusion = sigma_s * dW

        jump = 0.0
        if i in event_steps:
            J = rng.normal(mu_J, sigma_J)
            jump = J  # log-return jump

        log_S += drift + diffusion + jump
        log_S_path.append(log_S)

    return t, np.exp(np.array(log_S_path)), events


# ── Simulate multiple scenarios ────────────────────────────────────────────
HAWKES_PARAMS = dict(
    S0=100, mu_s=0.05, sigma_s=0.15,
    mu_J=-0.04, sigma_J=0.03,      # negative mean jump (crashes)
    hawkes_mu=3.0, alpha_h=0.6, beta_h=2.0,
    T=1.0, dt=1/252,
)

print(f"Stationarity: α/β = {HAWKES_PARAMS['alpha_h']/HAWKES_PARAMS['beta_h']:.3f} < 1 → {'OK' if HAWKES_PARAMS['alpha_h']/HAWKES_PARAMS['beta_h'] < 1 else 'NON-STATIONARY'}")

# Run several paths
paths_data = [simulate_hawkes_jump_diffusion(**HAWKES_PARAMS, seed=s) for s in range(6)]

In [ ]:
# ── Figure 4: Hawkes Jump-Diffusion ───────────────────────────────────────
fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(3, 2, hspace=0.45, wspace=0.35)

# ① Multi-path price evolution
ax1 = fig.add_subplot(gs[0, :])
for k, (t, S, ev) in enumerate(paths_data):
    alpha_p = 0.7 if k == 0 else 0.35
    lw_p    = 1.4 if k == 0 else 0.8
    color   = '#d62728' if k == 0 else cm.Blues(0.3 + 0.1*k)
    ax1.plot(t * 252, S, lw=lw_p, alpha=alpha_p, color=color,
             label='Representative path' if k == 0 else None)
    if k == 0:
        for ev_t in ev:
            ax1.axvline(ev_t * 252, ls=':', lw=0.5, color='orange', alpha=0.6)
ax1.set_ylabel('Price $S_t$')
ax1.set_title('Hawkes Jump-Diffusion: Price Paths (orange dashes = jump events)')
ax1.legend(); ax1.grid(alpha=0.2)

# ② Conditional intensity λ(t) for reference path
ax2 = fig.add_subplot(gs[1, 0])
t_ref, S_ref, ev_ref = paths_data[0]
t_grid_fine = np.linspace(0, 1, 2000)
lam_path = hawkes_intensity(ev_ref, HAWKES_PARAMS['hawkes_mu'],
                            HAWKES_PARAMS['alpha_h'], HAWKES_PARAMS['beta_h'],
                            t_grid_fine)
ax2.fill_between(t_grid_fine * 252, lam_path, alpha=0.35, color='#ff7f0e')
ax2.plot(t_grid_fine * 252, lam_path, lw=1.2, color='#d62728')
ax2.axhline(HAWKES_PARAMS['hawkes_mu'], ls='--', lw=1, color='navy',
            label=r'Baseline $\mu$')
ax2.set_xlabel('Day'); ax2.set_ylabel(r'$\lambda(t)$')
ax2.set_title('Conditional Jump Intensity')
ax2.legend(); ax2.grid(alpha=0.2)

# ③ Clustered jump times
ax3 = fig.add_subplot(gs[1, 1])
all_events = [e for _, _, e in paths_data]
for row_k, ev in enumerate(all_events):
    ax3.scatter(ev * 252, np.full_like(ev, row_k), s=12,
                color=cm.plasma(0.2 + 0.12*row_k), alpha=0.8)
ax3.set_xlabel('Day'); ax3.set_ylabel('Path index')
ax3.set_title('Jump Event Times (Clustered Contagion)')
ax3.grid(alpha=0.2)

# ④ Excitation parameter sweep
ax4 = fig.add_subplot(gs[2, 0])
for alpha_sweep, c in zip([0.1, 0.4, 0.7, 0.9], ['#1f77b4','#2ca02c','#ff7f0e','#d62728']):
    ev_s = simulate_hawkes(HAWKES_PARAMS['hawkes_mu'], alpha_sweep,
                           HAWKES_PARAMS['beta_h'], T=1.0, seed=0)
    lam  = hawkes_intensity(ev_s, HAWKES_PARAMS['hawkes_mu'], alpha_sweep,
                            HAWKES_PARAMS['beta_h'], t_grid_fine)
    ax4.plot(t_grid_fine * 252, lam, lw=1.1, color=c,
             label=fr'$\alpha_H={alpha_sweep}$ ({len(ev_s)} events)')
ax4.set_xlabel('Day'); ax4.set_ylabel(r'$\lambda(t)$')
ax4.set_title('Excitation Parameter Sensitivity')
ax4.legend(fontsize=8); ax4.grid(alpha=0.2)

# ⑤ Return distribution vs Gaussian
ax5 = fig.add_subplot(gs[2, 1])
all_rets = []
for s_seed in range(200):
    _, S_r, _ = simulate_hawkes_jump_diffusion(**HAWKES_PARAMS, seed=s_seed)
    all_rets.append(np.log(S_r[-1] / S_r[0]))
all_rets = np.array(all_rets)
ax5.hist(all_rets, bins=40, density=True, alpha=0.65,
         color='#d62728', label='Hawkes JD returns')
x_range = np.linspace(all_rets.min(), all_rets.max(), 200)
ax5.plot(x_range, norm.pdf(x_range, all_rets.mean(), all_rets.std()),
         lw=2, color='navy', label='Gaussian fit')
ax5.set_xlabel('Log-return'); ax5.set_ylabel('Density')
ax5.set_title('Return Distribution: Fat Tails via Jumps')
ax5.legend(); ax5.grid(alpha=0.2)

plt.savefig('hawkes_jump_diffusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → hawkes_jump_diffusion.png')

---
## Section 5 — Model Taxonomy Decision Matrix

The following table contextualises the five models in the engine's unified framework.

| Model | Variance Process | Key Strength | Primary Use Case | Limitation vs. Rough Volterra |
|---|---|---|---|---|
| **Rough Volterra** (Project) | Non-Markovian, $H\approx0.07$ | Short-term ATM IV skew; SPX term structure | Equity index vol arb | Path simulation cost |
| **Heston** | CIR (mean-reverting diffusion) | Closed-form char. fn.; hedging tractability | Equity structured products | $H=0.5$ too smooth |
| **SABR** | Correlated log-normal; CEV backbone | IV smile analytic formula | IR swaptions, FX options | No mean reversion |
| **Local Volatility** | Deterministic $\sigma(t,S)$ | Perfect arbitrage-free surface fit | Barrier/Asian/path-dep. exotics | Zero forward vol smile |
| **Hawkes JD** | Jump intensity self-excitation | Contagion and clustering | Credit, crisis modeling | No continuous vol |

**HMM ↔ Hawkes Connection:** The G-HMM traffic-light in the engine detects *macro* regime transitions (calm → turbulent), while the Hawkes process models the *micro* clustering of jump events within the turbulent state. These are complementary: the HMM provides the prior probability of entering a high-intensity regime, and the Hawkes kernel governs the cascade dynamics once a crisis is underway.

In [ ]:
# ── Figure 5: HMM-Hawkes integration conceptual diagram ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ① HMM posterior probability (simulated)
ax = axes[0]
np.random.seed(1)
T_hmm = 500
# Synthetic turbulent episodes
states = np.zeros(T_hmm, dtype=int)
for start, end in [(120, 160), (280, 320), (420, 460)]:
    states[start:end] = 1
posterior = np.where(states == 1,
                     0.55 + 0.44 * np.clip(np.random.beta(8, 2, T_hmm), 0, 1),
                     0.05 + 0.35 * np.clip(np.random.beta(2, 8, T_hmm), 0, 1))
posterior = np.clip(posterior, 0, 1)

ax.fill_between(range(T_hmm), posterior, alpha=0.35, color='#d62728',
                where=posterior > 0.6, label='Turbulent regime (P>0.6)')
ax.fill_between(range(T_hmm), posterior, alpha=0.2, color='#1f77b4',
                where=posterior <= 0.6)
ax.plot(posterior, lw=1, color='black', alpha=0.7)
ax.axhline(0.6, ls='--', lw=1.5, color='#d62728', label='Threshold 0.6')
ax.axhline(0.8, ls=':', lw=1.5, color='darkred', label='Close-all 0.8')
ax.set_xlabel('Trading Day')
ax.set_ylabel(r'$P(s_t = \mathrm{Turbulent} \mid r_{1:t})$')
ax.set_title('G-HMM: Forward-Filtered Posterior')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

# ② Hawkes intensity conditioned on HMM turbulent episodes
ax2 = axes[1]
t_h = np.linspace(0, 2, 3000)
# Simulate Hawkes only during turbulent periods
for i, (period_start, period_end, label) in enumerate([
    (0.24, 0.32, 'Crisis 1'),
    (0.56, 0.64, 'Crisis 2'),
    (0.84, 0.92, 'Crisis 3')
]):
    ev_crisis = simulate_hawkes(10.0, 0.7, 3.0, 0.08, seed=i*7)
    ev_shifted = ev_crisis + period_start
    lam_crisis = hawkes_intensity(ev_shifted[ev_shifted < period_end],
                                  2.0, 0.7, 3.0,
                                  t_h[(t_h >= period_start) & (t_h < period_end+0.05)])
    t_slice = t_h[(t_h >= period_start) & (t_h < period_end+0.05)]
    ax2.fill_between(t_slice, lam_crisis, alpha=0.4, color=cm.Reds(0.4 + 0.2*i))
    ax2.plot(t_slice, lam_crisis, lw=1.3, color=cm.Reds(0.5 + 0.2*i), label=label)

ax2.set_xlabel('Time (years)')
ax2.set_ylabel(r'Hawkes $\lambda(t)$')
ax2.set_title('Hawkes Intensity: Self-Exciting Cascades in Turbulent Regimes')
ax2.legend(fontsize=8); ax2.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('hmm_hawkes_integration.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → hmm_hawkes_integration.png')

---
## Summary

| Figure | File | Description |
|---|---|---|
| Heston Vol Surface | `heston_surface.png` | 3D IV surface + variance paths showing leverage effect |
| SABR Smile | `sabr_smile.png` | IV smile/skew at single maturity + ATM term structure |
| Dupire Local Vol | `dupire_local_vol.png` | Synthetic IV surface vs. calibrated local vol surface |
| Hawkes JD | `hawkes_jump_diffusion.png` | Price paths, intensity, clustering, fat-tail returns |
| HMM-Hawkes | `hmm_hawkes_integration.png` | Macro HMM posterior driving micro Hawkes cascades |

All figures are production-ready for direct inclusion in the LaTeX manuscript via `\includegraphics`.